<a href="https://colab.research.google.com/github/EDOETE/mini_projet_spark/blob/develop/Copie_de_mini_projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
#import necessary lybrary
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

In [ ]:
#creat pyspark session
spark = (
    SparkSession.builder
    .appName("FootballStats")
    .getOrCreate()
)

In [ ]:
# Afficher le nom de l'application
print("Nom de l'application :", spark.sparkContext.appName)

# Afficher la version de Spark
print("Version Spark :", spark.version)


Nom de l'application : FootballStats
Version Spark : 4.0.5


In [ ]:
#load the dataset
df_match=spark.read.format('csv').options(header='True').load('/content/drive/MyDrive/Colab Notebooks/data/football_matches.csv')


In [ ]:
#preview  of the data
df_match.limit(20).show()

+--------+---+------+----------+------------------+--------------+----+----+---+
|Match_ID|Div|Season|      Date|          HomeTeam|      AwayTeam|FTHG|FTAG|FTR|
+--------+---+------+----------+------------------+--------------+----+----+---+
|       1| D2|  2009|2010-04-04|        Oberhausen|Kaiserslautern|   2|   1|  H|
|       2| D2|  2009|2009-11-01|       Munich 1860|Kaiserslautern|   0|   1|  A|
|       3| D2|  2009|2009-10-04|     Frankfurt FSV|Kaiserslautern|   1|   1|  D|
|       4| D2|  2009|2010-02-21|     Frankfurt FSV|     Karlsruhe|   2|   1|  H|
|       5| D2|  2009|2009-12-06|             Ahlen|     Karlsruhe|   1|   3|  A|
|       6| D2|  2009|2010-04-03|      Union Berlin|     Karlsruhe|   1|   1|  D|
|       7| D2|  2009|2009-08-14|         Paderborn|     Karlsruhe|   2|   0|  H|
|       8| D2|  2009|2010-03-08|         Bielefeld|     Karlsruhe|   0|   1|  A|
|       9| D2|  2009|2009-09-26|    Kaiserslautern|     Karlsruhe|   2|   0|  H|
|      10| D2|  2009|2009-11

In [ ]:
#rename colum name
df_match=df_match.selectExpr(
    "*",
    "FTHG as HomeGoals",
    "FTAG as AwayGoals",
    "FTR as  MatchResult"

)

In [ ]:
df_match.limit(20).show()

+--------+---+------+----------+------------------+--------------+----+----+---+---------+---------+-----------+
|Match_ID|Div|Season|      Date|          HomeTeam|      AwayTeam|FTHG|FTAG|FTR|HomeGoals|AwayGoals|MatchResult|
+--------+---+------+----------+------------------+--------------+----+----+---+---------+---------+-----------+
|       1| D2|  2009|2010-04-04|        Oberhausen|Kaiserslautern|   2|   1|  H|        2|        1|          H|
|       2| D2|  2009|2009-11-01|       Munich 1860|Kaiserslautern|   0|   1|  A|        0|        1|          A|
|       3| D2|  2009|2009-10-04|     Frankfurt FSV|Kaiserslautern|   1|   1|  D|        1|        1|          D|
|       4| D2|  2009|2010-02-21|     Frankfurt FSV|     Karlsruhe|   2|   1|  H|        2|        1|          H|
|       5| D2|  2009|2009-12-06|             Ahlen|     Karlsruhe|   1|   3|  A|        1|        3|          A|
|       6| D2|  2009|2010-04-03|      Union Berlin|     Karlsruhe|   1|   1|  D|        1|      

In [ ]:
#delate colomn
df_match = df_match.drop(
  "FTHG","FTAG","FTR"
)


In [ ]:
df_match.limit(20).show()

+--------+---+------+----------+------------------+--------------+---------+---------+-----------+
|Match_ID|Div|Season|      Date|          HomeTeam|      AwayTeam|HomeGoals|AwayGoals|MatchResult|
+--------+---+------+----------+------------------+--------------+---------+---------+-----------+
|       1| D2|  2009|2010-04-04|        Oberhausen|Kaiserslautern|        2|        1|          H|
|       2| D2|  2009|2009-11-01|       Munich 1860|Kaiserslautern|        0|        1|          A|
|       3| D2|  2009|2009-10-04|     Frankfurt FSV|Kaiserslautern|        1|        1|          D|
|       4| D2|  2009|2010-02-21|     Frankfurt FSV|     Karlsruhe|        2|        1|          H|
|       5| D2|  2009|2009-12-06|             Ahlen|     Karlsruhe|        1|        3|          A|
|       6| D2|  2009|2010-04-03|      Union Berlin|     Karlsruhe|        1|        1|          D|
|       7| D2|  2009|2009-08-14|         Paderborn|     Karlsruhe|        2|        0|          H|
|       8|

In [ ]:
#Create Binary Columns
df_match = df_match.withColumn(
    "HomeTeamWin",
    when(col("MatchResult") == "H", 1).otherwise(0)
).withColumn(
    "AwayTeamWin",
    when(col("MatchResult") == "A", 1).otherwise(0)
).withColumn(
    "GameTie",
    when(col("MatchResult") == "D", 1).otherwise(0)
)

In [ ]:
df_match.limit(20).show()

+--------+---+------+----------+------------------+--------------+---------+---------+-----------+-----------+-----------+-------+
|Match_ID|Div|Season|      Date|          HomeTeam|      AwayTeam|HomeGoals|AwayGoals|MatchResult|HomeTeamWin|AwayTeamWin|GameTie|
+--------+---+------+----------+------------------+--------------+---------+---------+-----------+-----------+-----------+-------+
|       1| D2|  2009|2010-04-04|        Oberhausen|Kaiserslautern|        2|        1|          H|          1|          0|      0|
|       2| D2|  2009|2009-11-01|       Munich 1860|Kaiserslautern|        0|        1|          A|          0|          1|      0|
|       3| D2|  2009|2009-10-04|     Frankfurt FSV|Kaiserslautern|        1|        1|          D|          0|          0|      1|
|       4| D2|  2009|2010-02-21|     Frankfurt FSV|     Karlsruhe|        2|        1|          H|          1|          0|      0|
|       5| D2|  2009|2009-12-06|             Ahlen|     Karlsruhe|        1|       

In [ ]:
df_match_filtered = df_match.filter(
    (col("Div") == "D1") &
    (col("Season") >= 2013) &
    (col("Season") <= 2015)
)

In [ ]:
df_match_filtered.limit(10).toPandas()

,Match_ID,Div,Season,Date,HomeTeam,AwayTeam,HomeGoals,AwayGoals,MatchResult,HomeTeamWin,AwayTeamWin,GameTie
0,2316,D1,2013,2014-02-23,Hannover,Bayern Munich,0,4,A,0,1,0
1,2317,D1,2013,2013-11-02,Hoffenheim,Bayern Munich,1,2,A,0,1,0
2,2318,D1,2013,2014-01-29,Stuttgart,Bayern Munich,1,2,A,0,1,0
3,2319,D1,2013,2014-03-22,Mainz,Bayern Munich,0,2,A,0,1,0
4,2320,D1,2013,2014-03-25,Hertha,Bayern Munich,1,3,A,0,1,0
5,2321,D1,2013,2014-03-08,Wolfsburg,Bayern Munich,1,6,A,0,1,0
6,2322,D1,2013,2013-08-27,Freiburg,Bayern Munich,1,1,D,0,0,1
7,2323,D1,2013,2013-11-23,Dortmund,Bayern Munich,0,3,A,0,1,0
8,2324,D1,2013,2014-02-08,Nurnberg,Bayern Munich,0,2,A,0,1,0
9,2325,D1,2013,2014-04-19,Braunschweig,Bayern Munich,0,2,A,0,1,0


#Étape 7 — Statistiques à domicile

In [ ]:

df_home_stats = df_match_filtered.groupBy(
    col("Season"),
    col("HomeTeam")
)

In [ ]:
df_home_stats = df_home_stats.agg(
    sum(when(col("MatchResult") == "H", 1).otherwise(0)).alias("Wins"),
    sum(when(col("MatchResult") == "A", 1).otherwise(0)).alias("Losses"),
    sum(when(col("MatchResult") == "D", 1).otherwise(0)).alias("Draws"),
    sum(col("HomeGoals")).alias("GoalsScored"),
    sum(col("AwayGoals")).alias("GoalsConceded")
)

In [ ]:
df_home_stats.show(20)

+------+-------------+----+------+-----+-----------+-------------+
|Season|     HomeTeam|Wins|Losses|Draws|GoalsScored|GoalsConceded|
+------+-------------+----+------+-----+-----------+-------------+
|  2015|       Hertha|   9|     3|    5|       24.0|         15.0|
|  2014|    Paderborn|   4|     7|    6|       21.0|         31.0|
|  2015|Ein Frankfurt|   6|     5|    6|       22.0|         24.0|
|  2013|     Freiburg|   6|     7|    4|       25.0|         30.0|
|  2014|Werder Bremen|   8|     5|    4|       25.0|         24.0|
|  2014|        Mainz|   6|     5|    6|       27.0|         19.0|
|  2015|    Wolfsburg|   9|     3|    5|       32.0|         17.0|
|  2014|     Augsburg|   9|     4|    4|       28.0|         21.0|
|  2015|    Stuttgart|   6|    10|    1|       22.0|         32.0|
|  2015|   Hoffenheim|   6|     5|    6|       22.0|         25.0|
|  2013|Werder Bremen|   6|     5|    6|       21.0|         30.0|
|  2014|Ein Frankfurt|   9|     3|    5|       36.0|         2

In [ ]:
df_home_stats = df_home_stats.withColumnRenamed(
    "HomeTeam", "Team"
)

In [ ]:
df_home_stats.show(20)

+------+-------------+----+------+-----+-----------+-------------+
|Season|         Team|Wins|Losses|Draws|GoalsScored|GoalsConceded|
+------+-------------+----+------+-----+-----------+-------------+
|  2015|       Hertha|   9|     3|    5|       24.0|         15.0|
|  2014|    Paderborn|   4|     7|    6|       21.0|         31.0|
|  2015|Ein Frankfurt|   6|     5|    6|       22.0|         24.0|
|  2013|     Freiburg|   6|     7|    4|       25.0|         30.0|
|  2014|Werder Bremen|   8|     5|    4|       25.0|         24.0|
|  2014|        Mainz|   6|     5|    6|       27.0|         19.0|
|  2015|    Wolfsburg|   9|     3|    5|       32.0|         17.0|
|  2014|     Augsburg|   9|     4|    4|       28.0|         21.0|
|  2015|    Stuttgart|   6|    10|    1|       22.0|         32.0|
|  2015|   Hoffenheim|   6|     5|    6|       22.0|         25.0|
|  2013|Werder Bremen|   6|     5|    6|       21.0|         30.0|
|  2014|Ein Frankfurt|   9|     3|    5|       36.0|         2

#Étape 8 — Statistiques à l’extérieur

In [ ]:

df_home_stats = df_match_filtered.groupBy(
    col("Season"),
    col("HomeTeam")
)